# 05 - Limpieza inicial (de bronce a plata)

Acá cada tabla queda con los tipos correctos, solo con el periodo de estudio y con los valores inválidos en nulo. También se recupera el borough de los choques que no lo traen (numeral 7, como avance). Se lee de `/nyc/bronze/<fuente>` y se escribe en `/nyc/silver/<tabla>`.

En cada paso se anota cuántos registros quedan, y al final sale la tabla del embudo. No se inventa nada: lo inválido pasa a nulo y la imputación queda para la entrega final.

In [1]:
import pandas as pd
from pyspark.sql import functions as F, Window
from pyspark import StorageLevel

import geo
from codigos import (ARRESTOS_BORO, ARRESTOS_GRAVEDAD, ARRESTOS_GRUPOS_EDAD, DIAS_SEMANA,
                     POBREZA_BORO, POBREZA_EDUCACION, POBREZA_ESTADO, POBREZA_EDAD, POBREZA_ETNIA,
                     POBREZA_SEXO, SAT_BORO_DBN)
from config import BRONZE, SILVER, FECHA_INICIO, FECHA_FIN, sesion

spark = sesion("05_limpieza_inicial")
embudo = []

def paso(tabla, descripcion, df):
    n = df.count()
    embudo.append({"tabla": tabla, "paso": descripcion, "registros": n})
    print(f"{tabla:>10} | {descripcion:<55} {n:>12,}")

def mapa(diccionario):
    """dict de python a map de spark"""
    return F.create_map(*[F.lit(x) for kv in diccionario.items() for x in kv])

print("Periodo de estudio:", FECHA_INICIO, "a", FECHA_FIN)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/27 16:40:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Periodo de estudio: 2019-01-01 a 2026-06-30


## 1. PUMAs

Cargamos las 55 PUMAs y revisamos que estén bien: cuántas son, el borough según el código y que las coordenadas caigan en NYC. Si algo no cuadra, el notebook se detiene.

In [2]:
pumas = geo.cargar_pumas(spark)
pumas.groupby("borough")["puma"].agg(["count", "min", "max"])

,count,min,max
borough,,,
Bronx,10,3701,3710
Brooklyn,18,4001,4018
Manhattan,10,3801,3810
Queens,14,4101,4114
Staten Island,3,3901,3903


## 2. Arrestos

1. Se une el histórico con 2026. Antes del `unionByName` se renombran `Lon_Lat` y `Location` para que coincidan.
2. La fecha viene como `MM/dd/yyyy` y se pasa a tipo fecha.
3. Se deja solo lo que va del 2019-01-01 al 2026-06-30.
4. Se quitan los `ARREST_KEY` duplicados.
5. Las coordenadas fuera de Nueva York, incluidas las (0, 0), pasan a nulo.
6. Las edades que no son de las categorías publicadas (como `895`) pasan a nulo.
7. Se decodifican borough y gravedad: vacío o `(null)` queda como "Sin dato", y `9` o `I` como "Otro / no documentado".
8. Con el spatial join se ubica cada arresto en su PUMA.

In [3]:
hist = (spark.read.parquet(f"{BRONZE}/arrestos_historico")
        .withColumnRenamed("lon_lat", "punto_wkt").withColumn("fuente", F.lit("historico")))
ytd = (spark.read.parquet(f"{BRONZE}/arrestos_2026")
       .withColumnRenamed("location", "punto_wkt").withColumn("fuente", F.lit("2026")))
assert sorted(hist.columns) == sorted(ytd.columns), set(hist.columns) ^ set(ytd.columns)

arr = hist.unionByName(ytd)
paso("arrestos", "1. unión histórico + 2026", arr)

arr = arr.withColumn("fecha", F.to_date("arrest_date", "MM/dd/yyyy"))
sin_fecha = arr.filter(F.col("fecha").isNull()).count()
print(f"   fechas que no se pudieron convertir: {sin_fecha:,}")

arr = arr.filter(F.col("fecha").between(FECHA_INICIO, FECHA_FIN))
paso("arrestos", "2. dentro del periodo 2019-01-01 a 2026-06-30", arr)

arr = arr.dropDuplicates(["arrest_key"])
paso("arrestos", "3. sin ARREST_KEY duplicados", arr)

  arrestos | 1. unión histórico + 2026                                  6,406,848


   fechas que no se pudieron convertir: 0


  arrestos | 2. dentro del periodo 2019-01-01 a 2026-06-30              1,608,509


  arrestos | 3. sin ARREST_KEY duplicados                               1,608,509


In [4]:
lat, lon = F.col("latitude").cast("double"), F.col("longitude").cast("double")
valida = geo.coordenada_valida(lat, lon)
cat = F.trim(F.col("law_cat_cd"))
gravedad = (F.when(cat.isNull() | cat.isin("", "(null)"), F.lit("Sin dato"))
             .otherwise(F.coalesce(mapa(ARRESTOS_GRAVEDAD)[cat], F.lit("Otro / no documentado"))))

arr = (arr
    .withColumn("lat", F.when(valida, lat))
    .withColumn("lon", F.when(valida, lon))
    .withColumn("grupo_edad", F.when(F.col("age_group").isin(ARRESTOS_GRUPOS_EDAD), F.col("age_group")))
    .withColumn("borough", mapa(ARRESTOS_BORO)[F.col("arrest_boro")])
    .withColumn("gravedad", gravedad)
    .withColumn("anio", F.year("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn("dia_semana", mapa(DIAS_SEMANA)[F.dayofweek("fecha")])
)
arr = geo.agregar_puma(arr, pumas, "lat", "lon", "puma")

arrestos = arr.select(
    "arrest_key", "fecha", "anio", "mes", "dia_semana",
    F.col("pd_cd").cast("int").alias("pd_cd"), "pd_desc",
    F.col("ky_cd").cast("int").alias("ky_cd"), "ofns_desc",
    "law_code", "law_cat_cd", "gravedad",
    F.col("arrest_boro").alias("boro_codigo"), "borough",
    F.col("arrest_precinct").cast("int").alias("precinto"),
    F.col("jurisdiction_code").cast("int").alias("jurisdiccion"),
    F.col("age_group").alias("age_group_original"), "grupo_edad",
    F.col("perp_sex").alias("sexo"), F.col("perp_race").alias("raza"),
    "lat", "lon", "puma", "fuente",
)
arrestos.write.mode("overwrite").partitionBy("anio").parquet(f"{SILVER}/arrestos")
arrestos = spark.read.parquet(f"{SILVER}/arrestos")
paso("arrestos", "4. escrito en plata", arrestos)

arrestos.agg(
    F.sum(F.col("lat").isNull().cast("int")).alias("sin_coordenada_valida"),
    F.sum(F.col("grupo_edad").isNull().cast("int")).alias("edad_invalida_o_vacia"),
    F.sum(F.col("puma").isNull().cast("int")).alias("sin_puma"),
    F.sum((F.col("gravedad") == "Otro / no documentado").cast("int")).alias("gravedad_no_documentada"),
    F.sum((F.col("gravedad") == "Sin dato").cast("int")).alias("gravedad_sin_dato"),
).toPandas()

26/09/27 16:41:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


  arrestos | 4. escrito en plata                                        1,608,509


,sin_coordenada_valida,edad_invalida_o_vacia,sin_puma,gravedad_no_documentada,gravedad_sin_dato
0,15,147237,201,4963,11240


Como control, comparamos el borough que puso la policía con el que sale de la PUMA.

In [5]:
(arrestos.filter(F.col("puma").isNotNull())
 .withColumn("borough_puma", geo.borough_desde_puma(F.col("puma")))
 .agg(F.avg((F.col("borough") == F.col("borough_puma")).cast("int")).alias("coincidencia"))
 .withColumn("coincidencia", F.round(F.col("coincidencia") * 100, 2))
 .toPandas())

,coincidencia
0,99.7


## 3. Choques

1. Se sacan la fecha, la hora y el día de la semana (`CRASH TIME` viene como `H:mm`).
2. Se corta en el último mes completo, porque el dataset solo llega hasta mediados de junio de 2026.
3. Los conteos se pasan a enteros. Un choque es grave si tiene por lo menos un herido o un muerto.
4. Las coordenadas fuera de Nueva York pasan a nulo.
5. Si falta `BOROUGH` pero hay coordenadas, se usa el de la PUMA. La columna `fuente_borough` dice de dónde salió.

In [6]:
ch = spark.read.parquet(f"{BRONZE}/choques")
paso("choques", "1. registros originales", ch)

ch = ch.withColumn("fecha", F.to_date("crash_date", "MM/dd/yyyy"))
fecha_max = ch.agg(F.max("fecha")).first()[0]
# último día del mes anterior a la fecha máxima
corte = min(pd.Timestamp(fecha_max).replace(day=1) - pd.Timedelta(days=1), pd.Timestamp(FECHA_FIN)).date()
print(f"Fecha máxima en los datos: {fecha_max}  ->  corte fijo del análisis: {corte}")

ch = ch.filter(F.col("fecha").between(FECHA_INICIO, str(corte)))
paso("choques", f"2. dentro del periodo {FECHA_INICIO} a {corte}", ch)

ch = ch.dropDuplicates(["collision_id"])
paso("choques", "3. sin COLLISION_ID duplicados", ch)

   choques | 1. registros originales                                    2,269,187


Fecha máxima en los datos: 2026-06-11  ->  corte fijo del análisis: 2026-05-31


   choques | 2. dentro del periodo 2019-01-01 a 2026-05-31                846,420


   choques | 3. sin COLLISION_ID duplicados                               846,420


In [7]:
conteos = {c: F.col(c).cast("int") for c in [
    "number_of_persons_injured", "number_of_persons_killed",
    "number_of_pedestrians_injured", "number_of_pedestrians_killed",
    "number_of_cyclist_injured", "number_of_cyclist_killed",
    "number_of_motorist_injured", "number_of_motorist_killed"]}
lat, lon = F.col("latitude").cast("double"), F.col("longitude").cast("double")
valida = geo.coordenada_valida(lat, lon)
hora = F.split(F.col("crash_time"), ":")

ch = (ch
    .withColumn("hora", hora.getItem(0).cast("int"))
    .withColumn("minuto", hora.getItem(1).cast("int"))
    .withColumn("anio", F.year("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn("dia_semana", mapa(DIAS_SEMANA)[F.dayofweek("fecha")])
    .withColumn("lat", F.when(valida, lat))
    .withColumn("lon", F.when(valida, lon))
    .withColumn("borough_original", F.when(F.trim("borough") != "", F.initcap(F.trim("borough"))))
)
for nombre, expr in conteos.items():
    ch = ch.withColumn(nombre, expr)
ch = ch.withColumn("heridos", F.col("number_of_persons_injured")).withColumn("muertos", F.col("number_of_persons_killed"))
ch = ch.withColumn("grave", (F.coalesce("heridos", F.lit(0)) + F.coalesce("muertos", F.lit(0))) > 0)

ch = geo.agregar_puma(ch, pumas, "lat", "lon", "puma")
ch = (ch
    .withColumn("borough_puma", geo.borough_desde_puma(F.col("puma")))
    .withColumn("borough", F.coalesce("borough_original", "borough_puma"))
    .withColumn("fuente_borough",
        F.when(F.col("borough_original").isNotNull(), "original")
         .when(F.col("borough_puma").isNotNull(), "recuperado por coordenadas")
         .otherwise("sin dato"))
)

choques = ch.select(
    F.col("collision_id").cast("long").alias("collision_id"), "fecha", "anio", "mes", "dia_semana", "hora", "minuto",
    "borough", "fuente_borough", "borough_original", "puma", "lat", "lon",
    F.col("zip_code").alias("codigo_postal"),
    "heridos", "muertos", "grave", *[c for c in conteos if c not in ("number_of_persons_injured", "number_of_persons_killed")],
    *[F.col(f"contributing_factor_vehicle_{i}").alias(f"factor_{i}") for i in range(1, 6)],
    *[F.col(f"vehicle_type_code_{i}").alias(f"tipo_vehiculo_{i}") for i in range(1, 6)],
)
choques.write.mode("overwrite").partitionBy("anio").parquet(f"{SILVER}/choques")
choques = spark.read.parquet(f"{SILVER}/choques")
paso("choques", "4. escrito en plata", choques)

(choques.groupBy("fuente_borough").count()
 .withColumn("porcentaje", F.round(100 * F.col("count") / F.sum("count").over(Window.partitionBy()), 2))
 .orderBy(F.desc("count")).toPandas())

   choques | 4. escrito en plata                                          846,420


26/09/27 16:42:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/27 16:42:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/09/27 16:42:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/27 16:42:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/27 16:42:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/27 16:42:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


,fuente_borough,count,porcentaje
0,original,577179,68.19
1,recuperado por coordenadas,219488,25.93
2,sin dato,49753,5.88


Los "recuperado por coordenadas" son choques que venían sin borough. Los "sin dato" no tienen ni borough ni coordenadas, así que no entran en los análisis por borough.

Igual que en arrestos, comparamos el borough registrado con el de la PUMA.

In [8]:
(choques.filter(F.col("borough_original").isNotNull() & F.col("puma").isNotNull())
 .withColumn("borough_puma", geo.borough_desde_puma(F.col("puma")))
 .agg(F.round(100 * F.avg((F.col("borough_original") == F.col("borough_puma")).cast("int")), 2).alias("coincidencia"))
 .toPandas())

,coincidencia
0,99.88


## 4. Vehículos

Solo se dejan los vehículos de choques del mismo periodo, y se unen por `COLLISION_ID`.
- Si `VEHICLE_YEAR` no está entre 1900 y 2027, pasa a nulo.
- `tiene_choque` dice si el choque de ese vehículo está en la tabla limpia de choques.

In [9]:
ve = spark.read.parquet(f"{BRONZE}/vehiculos")
paso("vehiculos", "1. registros originales", ve)

ve = (ve.withColumn("fecha", F.to_date("crash_date", "MM/dd/yyyy"))
        .filter(F.col("fecha").between(FECHA_INICIO, str(corte))))
paso("vehiculos", f"2. dentro del periodo {FECHA_INICIO} a {corte}", ve)
ve = ve.dropDuplicates(["unique_id"])
paso("vehiculos", "3. sin UNIQUE_ID duplicados", ve)

anio_v = F.col("vehicle_year").cast("int")
ids_choques = choques.select("collision_id").withColumn("tiene_choque", F.lit(True))
vehiculos = (ve
    .withColumn("collision_id", F.col("collision_id").cast("long"))
    .join(ids_choques, "collision_id", "left")
    .select(
        F.col("unique_id").cast("long").alias("unique_id"), "collision_id", "fecha",
        F.year("fecha").alias("anio"),
        F.col("vehicle_type").alias("tipo_vehiculo"),
        F.col("vehicle_make").alias("marca"),
        F.when(anio_v.between(1900, 2027), anio_v).alias("anio_vehiculo"),
        F.col("vehicle_occupants").cast("int").alias("ocupantes"),
        F.col("driver_sex").alias("sexo_conductor"),
        F.col("driver_license_status").alias("licencia"),
        F.col("pre_crash").alias("maniobra_previa"),
        F.col("point_of_impact").alias("punto_impacto"),
        F.col("contributing_factor_1").alias("factor_1"),
        F.col("contributing_factor_2").alias("factor_2"),
        F.coalesce("tiene_choque", F.lit(False)).alias("tiene_choque"),
    ))
vehiculos.write.mode("overwrite").partitionBy("anio").parquet(f"{SILVER}/vehiculos")
vehiculos = spark.read.parquet(f"{SILVER}/vehiculos")
paso("vehiculos", "4. escrito en plata", vehiculos)
vehiculos.groupBy("tiene_choque").count().toPandas()

 vehiculos | 1. registros originales                                    4,551,002


 vehiculos | 2. dentro del periodo 2019-01-01 a 2026-05-31              1,710,836


 vehiculos | 3. sin UNIQUE_ID duplicados                                1,710,836


 vehiculos | 4. escrito en plata                                        1,710,836


,tiene_choque,count
0,True,1710836


## 5. Pobreza (ACS 2018)

Los códigos se traducen con el diccionario (`notebooks/codigos.py`). Acá no se filtra por periodo porque es una sola encuesta, la de 2018. Las tasas se sacan ponderando por `PWGTP`.

In [10]:
po = spark.read.parquet(f"{BRONZE}/pobreza")
paso("pobreza", "1. registros originales", po)
pobreza = po.select(
    F.col("serialno").cast("long").alias("hogar"),
    F.col("sporder").cast("int").alias("persona"),
    F.col("pwgtp").cast("int").alias("peso_persona"),
    F.col("wgtp").cast("int").alias("peso_hogar"),
    F.col("agep").cast("int").alias("edad"),
    mapa(POBREZA_EDAD)[F.col("agecateg").cast("int")].alias("grupo_edad"),
    mapa(POBREZA_BORO)[F.col("boro").cast("int")].alias("borough"),
    mapa(POBREZA_EDUCACION)[F.col("educattain").cast("int")].alias("educacion"),
    F.col("educattain").cast("int").alias("educacion_codigo"),
    mapa(POBREZA_ETNIA)[F.col("ethnicity").cast("int")].alias("etnia"),
    mapa(POBREZA_SEXO)[F.col("sex").cast("int")].alias("sexo"),
    (F.col("nycgov_pov_stat").cast("int") == 1).alias("en_pobreza"),
    mapa(POBREZA_ESTADO)[F.col("nycgov_pov_stat").cast("int")].alias("estado_pobreza"),
    (F.col("off_pov_stat").cast("int") == 1).alias("en_pobreza_oficial"),
    F.col("nycgov_income").cast("double").alias("ingreso_nycgov"),
    F.col("nycgov_threshold").cast("double").alias("umbral_nycgov"),
)
pobreza.write.mode("overwrite").parquet(f"{SILVER}/pobreza")
pobreza = spark.read.parquet(f"{SILVER}/pobreza")
paso("pobreza", "2. escrito en plata", pobreza)
pobreza.groupBy("borough").agg(
    F.count("*").alias("personas_muestra"),
    F.sum("peso_persona").alias("poblacion_estimada"),
).orderBy("borough").toPandas()

   pobreza | 1. registros originales                                       68,273


   pobreza | 2. escrito en plata                                           68,273


,borough,personas_muestra,poblacion_estimada
0,Bronx,9806,1389245
1,Brooklyn,24184,2540638
2,Manhattan,9065,1568288
3,Queens,21198,2250749
4,Staten Island,4020,469393


`personas_muestra` es cuánta gente respondió la encuesta y `poblacion_estimada` (la suma de los pesos) es cuánta gente representan. La segunda es la que va en el denominador de las tasas.

## 6. SAT 2012

- Los puntajes `"s"` (suprimidos) quedan en nulo y se marcan con `suprimido`.
- El borough sale de la tercera letra del DBN.

In [11]:
sat = spark.read.parquet(f"{BRONZE}/sat_2012")
paso("sat", "1. registros originales", sat)
puntaje = lambda c: F.when(F.col(c) != "s", F.col(c).cast("int"))
sat = sat.select(
    "dbn", F.col("school_name").alias("colegio"),
    mapa(SAT_BORO_DBN)[F.substring("dbn", 3, 1)].alias("borough"),
    puntaje("num_of_sat_test_takers").alias("estudiantes"),
    puntaje("sat_critical_reading_avg_score").alias("lectura"),
    puntaje("sat_math_avg_score").alias("matematicas"),
    puntaje("sat_writing_avg_score").alias("escritura"),
    (F.col("sat_math_avg_score") == "s").alias("suprimido"),
).withColumn("total", F.col("lectura") + F.col("matematicas") + F.col("escritura"))
sat.write.mode("overwrite").parquet(f"{SILVER}/sat")
sat = spark.read.parquet(f"{SILVER}/sat")
paso("sat", "2. escrito en plata", sat)
sat.groupBy("borough").agg(F.count("*").alias("colegios"), F.sum(F.col("suprimido").cast("int")).alias("suprimidos"),
                           F.round(F.avg("total"), 1).alias("puntaje_total_promedio")).orderBy("borough").toPandas()

       sat | 1. registros originales                                          478


       sat | 2. escrito en plata                                              478


,borough,colegios,suprimidos,puntaje_total_promedio
0,Bronx,128,18,1144.6
1,Brooklyn,138,15,1167.4
2,Manhattan,121,15,1260.2
3,Queens,78,8,1275.4
4,Staten Island,13,1,1359.2


## 7. Embudo de limpieza

Cuántos registros quedan después de cada paso.

In [12]:
embudo_df = pd.DataFrame(embudo)
embudo_df.to_csv("../figuras/tabla_embudo_limpieza.csv", index=False)
embudo_df

,tabla,paso,registros
0,arrestos,1. unión histórico + 2026,6406848
1,arrestos,2. dentro del periodo 2019-01-01 a 2026-06-30,1608509
2,arrestos,3. sin ARREST_KEY duplicados,1608509
3,arrestos,4. escrito en plata,1608509
4,choques,1. registros originales,2269187
5,choques,2. dentro del periodo 2019-01-01 a 2026-05-31,846420
6,choques,3. sin COLLISION_ID duplicados,846420
7,choques,4. escrito en plata,846420
8,vehiculos,1. registros originales,4551002
9,vehiculos,2. dentro del periodo 2019-01-01 a 2026-05-31,1710836


## Conclusiones

- Arrestos: se juntaron las dos tablas, se filtró el periodo, se quitaron duplicados y los valores imposibles quedaron en nulo.
- Choques: se cortó en el último mes completo y se recuperó el borough con las coordenadas.
- Vehículos: años imposibles en nulo y marcados los vehículos que no tienen choque.
- Pobreza y SAT: decodificados con su documentación, y los puntajes `s` en nulo.